In [ ]:
!pip install tabulate

In [ ]:
import sqlite3
import re
import os
from tabulate import tabulate
from datetime import date
from google.colab import files

# UPLOAD SQL FILE
print("Please upload your biosync_project.sql file...")
uploaded = files.upload()
sql_filename = list(uploaded.keys())[0]

with open(sql_filename, 'r') as f:
    sql_content = f.read()

print(f"Uploaded: {sql_filename}")

# CREATE FRESH DATABASE
if os.path.exists('biosync.db'):
    os.remove('biosync.db')
    print("Old database cleared")

conn = sqlite3.connect('biosync.db')
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = OFF")

cursor.executescript("""
CREATE TABLE IF NOT EXISTS Patient (
    patient_id   TEXT PRIMARY KEY,
    first_name   TEXT NOT NULL,
    last_name    TEXT NOT NULL,
    dob          TEXT NOT NULL,
    blood_group  TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Doctor (
    doctor_id      TEXT PRIMARY KEY,
    first_name     TEXT NOT NULL,
    last_name      TEXT NOT NULL,
    license_no     TEXT NOT NULL UNIQUE,
    specialization TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Medication (
    drug_id       TEXT PRIMARY KEY,
    name          TEXT NOT NULL UNIQUE,
    gene_pathway  TEXT NOT NULL,
    standard_dose TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Medication_Side_Effects (
    drug_id     TEXT NOT NULL,
    side_effect TEXT NOT NULL,
    PRIMARY KEY (drug_id, side_effect)
);
CREATE TABLE IF NOT EXISTS Genomic_Profile (
    profile_id TEXT PRIMARY KEY,
    seq_id     TEXT NOT NULL UNIQUE,
    variant    TEXT NOT NULL,
    patient_id TEXT NOT NULL UNIQUE,
    test_date  TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Genomic_Drug_Compatibility (
    compatibility_id TEXT PRIMARY KEY,
    variant          TEXT NOT NULL,
    gene_pathway     TEXT NOT NULL,
    compatibility    TEXT NOT NULL,
    reason           TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Safety_Validation (
    validation_id   TEXT PRIMARY KEY,
    validation_date TEXT NOT NULL,
    status          TEXT NOT NULL,
    patient_id      TEXT NOT NULL,
    doctor_id       TEXT NOT NULL,
    drug_id         TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Clinical_Log (
    log_id         TEXT PRIMARY KEY,
    log_date       TEXT NOT NULL,
    heart_rate     INTEGER NOT NULL,
    temperature    REAL NOT NULL,
    blood_pressure TEXT NOT NULL,
    outcome        TEXT NOT NULL,
    patient_id     TEXT NOT NULL,
    doctor_id      TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS Log_Symptoms (
    log_id  TEXT NOT NULL,
    symptom TEXT NOT NULL,
    PRIMARY KEY (log_id, symptom)
);
""")
conn.commit()
print("All tables created")

# LOAD DATA
sql_content = re.sub(r"DATE '(\d{4}-\d{2}-\d{2})'", r"'\1'", sql_content)
lines = sql_content.split('\n')
insert_lines = [l.strip() for l in lines if l.strip().upper().startswith('INSERT INTO')]

counts = {}
for line in insert_lines:
    match = re.match(r'INSERT INTO (\w+)', line, re.IGNORECASE)
    if not match:
        continue
    table = match.group(1)
    fixed = re.sub(r'INSERT INTO', 'INSERT OR IGNORE INTO', line, count=1, flags=re.IGNORECASE)
    try:
        cursor.execute(fixed)
        counts[table] = counts.get(table, 0) + 1
    except:
        pass

conn.commit()


# GENERATE SAFETY VALIDATIONS. Since SQLite doesn't support stored procedures,we replicate Regenerate_Safety_Validations() in Python

cursor.execute("DELETE FROM Safety_Validation")

cursor.execute("""
    SELECT p.patient_id, m.drug_id, gp.variant, m.gene_pathway
    FROM Patient p
    JOIN Genomic_Profile gp ON p.patient_id = gp.patient_id
    JOIN Medication m
    ORDER BY p.patient_id, m.drug_id
""")
pairs = cursor.fetchall()

doctors = [f"D{str(i).zfill(3)}" for i in range(1, 11)]
counter = 1

for patient_id, drug_id, variant, pathway in pairs:
    cursor.execute("""
        SELECT compatibility FROM Genomic_Drug_Compatibility
        WHERE variant = ? AND gene_pathway = ?
    """, (variant, pathway))
    row = cursor.fetchone()
    status = row[0] if row else 'Safe'

    doctor_id = doctors[(counter - 1) % 10]
    val_id = f"V{str(counter).zfill(4)}"

    import random
    days_ago = random.randint(1, 1000)
    val_date = date.fromordinal(date.today().toordinal() - days_ago).isoformat()

    try:
        cursor.execute("""
            INSERT OR IGNORE INTO Safety_Validation VALUES (?, ?, ?, ?, ?, ?)
        """, (val_id, val_date, status, patient_id, doctor_id, drug_id))
        counter += 1
    except:
        pass

conn.commit()
# MANUALLY INSERT GENOMIC_Drug_Compatibility RULES (multi-line SQL not caught by regex loader)
compatibility_rules = [
    ('GC001', 'CYP2C9*2',   'CYP2C9_substrate',   'Caution', 'Reduced CYP2C9 activity — lower dose needed'),
    ('GC002', 'CYP2C9*3',   'CYP2C9_substrate',   'Blocked', 'Severely reduced CYP2C9 — drug contraindicated'),
    ('GC003', 'CYP2C19*2',  'CYP2C19_substrate',  'Blocked', 'Poor metabolizer — drug ineffective or toxic'),
    ('GC004', 'CYP2C19*17', 'CYP2C19_substrate',  'Caution', 'Ultra-rapid metabolizer — higher dose may be needed'),
    ('GC005', 'CYP2D6*4',   'CYP2D6_substrate',   'Blocked', 'Poor metabolizer — drug accumulates to toxic levels'),
    ('GC006', 'CYP2D6*6',   'CYP2D6_substrate',   'Blocked', 'Poor metabolizer — drug accumulates to toxic levels'),
    ('GC007', 'TPMT*3A',    'TPMT_substrate',     'Blocked', 'Severely reduced TPMT — risk of fatal toxicity'),
    ('GC008', 'TPMT*3C',    'TPMT_substrate',     'Blocked', 'Reduced TPMT activity — dose reduction required'),
    ('GC009', 'SLCO1B1*5',  'SLCO1B1_substrate',  'Caution', 'Reduced drug transport — monitor for side effects'),
    ('GC010', 'MTHFR_677T', 'MTHFR_substrate',    'Caution', 'Reduced folate metabolism — supplement required'),
    ('GC011', 'BRCA1_mut',  'CYP2D6_substrate',   'Caution', 'BRCA1 mutation — increased cancer drug sensitivity'),
    ('GC012', 'Normal',     'CYP2C9_substrate',   'Safe',    'Normal metabolizer — standard dose applies'),
    ('GC013', 'Normal',     'CYP2C19_substrate',  'Safe',    'Normal metabolizer — standard dose applies'),
    ('GC014', 'Normal',     'CYP2D6_substrate',   'Safe',    'Normal metabolizer — standard dose applies'),
    ('GC015', 'Normal',     'TPMT_substrate',     'Safe',    'Normal metabolizer — standard dose applies'),
    ('GC016', 'Normal',     'SLCO1B1_substrate',  'Safe',    'Normal metabolizer — standard dose applies'),
    ('GC017', 'Normal',     'MTHFR_substrate',    'Safe',    'Normal metabolizer — standard dose applies'),
]

cursor.executemany("""
    INSERT OR IGNORE INTO Genomic_Drug_Compatibility
    VALUES (?, ?, ?, ?, ?)
""", compatibility_rules)
conn.commit()
print(f"Genomic_Drug_Compatibility → {len(compatibility_rules)} rows inserted")
#PRINT SUMMARY
print("\nData loaded:")
print("-" * 45)
for table in ['Patient','Doctor','Medication','Medication_Side_Effects',
              'Genomic_Profile','Genomic_Drug_Compatibility',
              'Safety_Validation','Clinical_Log','Log_Symptoms']:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"  {table:<35} → {cursor.fetchone()[0]} rows")
print("-" * 45)
# ============================================================
# RESET AND REGENERATE SAFETY VALIDATIONS CORRECTLY
# Run AFTER compatibility rules are inserted
# ============================================================
cursor.execute("DELETE FROM Safety_Validation")
conn.commit()

cursor.execute("""
    SELECT p.patient_id, m.drug_id, gp.variant, m.gene_pathway
    FROM Patient p
    JOIN Genomic_Profile gp ON p.patient_id = gp.patient_id
    JOIN Medication m
    ORDER BY p.patient_id, m.drug_id
""")
pairs = cursor.fetchall()

doctors = [f"D{str(i).zfill(3)}" for i in range(1, 11)]
counter = 1

for patient_id, drug_id, variant, pathway in pairs:
    cursor.execute("""
        SELECT compatibility FROM Genomic_Drug_Compatibility
        WHERE variant = ? AND gene_pathway = ?
    """, (variant, pathway))
    row = cursor.fetchone()
    status = row[0] if row else 'Safe'

    doctor_id = doctors[(counter - 1) % 10]
    val_id = f"V{str(counter).zfill(4)}"
    days_ago = random.randint(1, 1000)
    val_date = date.fromordinal(date.today().toordinal() - days_ago).isoformat()

    try:
        cursor.execute("""
            INSERT OR IGNORE INTO Safety_Validation
            VALUES (?, ?, ?, ?, ?, ?)
        """, (val_id, val_date, status, patient_id, doctor_id, drug_id))
        counter += 1
    except:
        pass

conn.commit()

# Verify
cursor.execute("SELECT status, COUNT(*) FROM Safety_Validation GROUP BY status")
rows = cursor.fetchall()
print("\n Safety_Validation breakdown:")
for row in rows:
    print(f"  {row[0]} → {row[1]}")
print("Database ready!\n")

Please upload your biosync_project.sql file...


Saving biosync project.sql to biosync project (1).sql
Uploaded: biosync project (1).sql
Old database cleared
All tables created
Genomic_Drug_Compatibility → 17 rows inserted

Data loaded:
---------------------------------------------
  Patient                             → 150 rows
  Doctor                              → 10 rows
  Medication                          → 8 rows
  Medication_Side_Effects             → 24 rows
  Genomic_Profile                     → 150 rows
  Genomic_Drug_Compatibility          → 17 rows
  Safety_Validation                   → 1200 rows
  Clinical_Log                        → 189 rows
  Log_Symptoms                        → 364 rows
---------------------------------------------

 Safety_Validation breakdown:
  Blocked → 125
  Caution → 72
  Safe → 1003
Database ready!



In [ ]:
import sqlite3
from tabulate import tabulate
from datetime import date

# Reconnect to the loaded database
conn = sqlite3.connect('biosync.db')
cursor = conn.cursor()
print("Connected to biosync.db")

# HELPER
def run_query(query, params=None, title=""):
    if title:
        print(f"\n{'='*55}")
        print(f"  {title}")
        print(f"{'='*55}")
    try:
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        rows = cursor.fetchall()
        cols = [desc[0] for desc in cursor.description]
        if rows:
            print(tabulate(rows, headers=cols, tablefmt="pretty"))
        else:
            print("No results found.")
        return rows
    except Exception as e:
        print(f"Error: {e}")
        return []

# FUNCTION 1 — Calculate Age
def calculate_age(patient_id):
    cursor.execute(
        "SELECT first_name, last_name, dob FROM Patient WHERE patient_id=?",
        (patient_id,)
    )
    row = cursor.fetchone()
    if not row:
        print("EXCEPTION: Patient not found.")
        return
    dob = date.fromisoformat(row[2])
    age = (date.today() - dob).days // 365
    print(f"\nPatient  : {row[0]} {row[1]}")
    print(f"Age: {age} years")

# FUNCTION 2 — Get Safety Score
def get_safety_score(patient_id, drug_id):
    cursor.execute("""
        SELECT sv.status, m.name, gp.variant
        FROM Safety_Validation sv
        JOIN Medication m ON sv.drug_id = m.drug_id
        JOIN Genomic_Profile gp ON sv.patient_id = gp.patient_id
        WHERE sv.patient_id = ? AND sv.drug_id = ?
        ORDER BY sv.validation_date DESC LIMIT 1
    """, (patient_id, drug_id))
    row = cursor.fetchone()
    if not row:
        print(" No validation record found. Score = -1")
        return
    score_map = {'Safe': 1, 'Caution': 2, 'Blocked': 3}
    score = score_map.get(row[0], -1)
    label = {1: "Safe", 2: "Caution", 3: "Blocked"}
    print(f"\n Drug     : {row[1]}")
    print(f" Variant  : {row[2]}")
    print(f" Score    : {score}  →  {label.get(score, 'Unknown')}")

# FUNCTION 3 — Get Genomic Compatibility (NEW)
def get_genomic_compatibility(patient_id, drug_id):
    cursor.execute(
        "SELECT variant FROM Genomic_Profile WHERE patient_id=?",
        (patient_id,)
    )
    row = cursor.fetchone()
    if not row:
        print("No genomic profile found for this patient.")
        return
    variant = row[0]

    cursor.execute(
        "SELECT gene_pathway FROM Medication WHERE drug_id=?",
        (drug_id,)
    )
    row = cursor.fetchone()
    if not row:
        print("Drug not found.")
        return
    pathway = row[0]

    cursor.execute("""
        SELECT compatibility, reason
        FROM Genomic_Drug_Compatibility
        WHERE variant = ? AND gene_pathway = ?
    """, (variant, pathway))
    row = cursor.fetchone()

    if row:
        label = {'Safe': 'Green', 'Caution': 'Yellow', 'Blocked': 'Red'}
        print(f"\n  Variant      : {variant}")
        print(f"  Pathway      : {pathway}")
        print(f"  {label.get(row[0],'')} Compatibility : {row[0]}")
        print(f"  Reason       : {row[1]}")
    else:
        print(f"\n  Variant  : {variant}")
        print(f" Pathway  : {pathway}")
        print(f"  Compatibility: Safe (no specific rule found)")

# PROCEDURE — Generate Log Entry
def generate_log_entry(log_id, patient_id, doctor_id,
                        heart_rate, temperature, bp, outcome):
    cursor.execute(
        "SELECT COUNT(*) FROM Patient WHERE patient_id=?", (patient_id,)
    )
    if cursor.fetchone()[0] == 0:
        print("EXCEPTION: Patient not found.")
        return

    cursor.execute(
        "SELECT COUNT(*) FROM Doctor WHERE doctor_id=?", (doctor_id,)
    )
    if cursor.fetchone()[0] == 0:
        print("EXCEPTION: Doctor not found.")
        return

    cursor.execute(
        "SELECT COUNT(*) FROM Genomic_Profile WHERE patient_id=?", (patient_id,)
    )
    if cursor.fetchone()[0] == 0:
        print("GENOMIC_DATA_MISSING: No genomic profile for this patient.")
        return

    cursor.execute(
        "SELECT COUNT(*) FROM Clinical_Log WHERE log_id=?", (log_id,)
    )
    if cursor.fetchone()[0] > 0:
        print("EXCEPTION: Duplicate log_id. Entry already exists.")
        return

    # Trigger simulation — check blocked validations
    cursor.execute("""
        SELECT COUNT(*) FROM Safety_Validation
        WHERE patient_id = ? AND status = 'Blocked'
    """, (patient_id,))
    if cursor.fetchone()[0] > 0:
        print("TRIGGER FIRED: SAFETY ALERT — Blocked validation exists for this patient.")
        print("   Cannot create clinical log until genomic profile is reviewed.")
        return

    try:
        cursor.execute("""
            INSERT INTO Clinical_Log
            (log_id, log_date, heart_rate, temperature,
             blood_pressure, outcome, patient_id, doctor_id)
            VALUES (?, DATE('now'), ?, ?, ?, ?, ?, ?)
        """, (log_id, heart_rate, temperature, bp, outcome, patient_id, doctor_id))
        conn.commit()
        print(f"Log '{log_id}' created successfully for patient {patient_id}!")
    except Exception as e:
        print(f"Error: {e}")

# CURSOR — Review Side Effects
def review_side_effects(patient_id):
    run_query("""
        SELECT DISTINCT m.drug_id, m.name AS DrugName,
               mse.side_effect AS SideEffect,
               sv.status AS RiskStatus
        FROM Safety_Validation sv
        JOIN Medication m ON sv.drug_id = m.drug_id
        JOIN Medication_Side_Effects mse ON m.drug_id = mse.drug_id
        WHERE sv.patient_id = ?
        ORDER BY sv.status DESC, m.name
    """, (patient_id,), title=f"Side Effects Report — {patient_id}")

# MENU SECTIONS
def menu_functions():
    print("\n--- FUNCTIONS ---")
    print("1. Calculate Patient Age")
    print("2. Get Safety Score")
    print("3. Get Genomic Compatibility (NEW)")
    choice = input("Choice: ").strip()
    if choice == "1":
        pid = input("  Patient ID (e.g. P001): ").strip()
        calculate_age(pid)
    elif choice == "2":
        pid = input("  Patient ID (e.g. P001): ").strip()
        did = input("  Drug ID   (e.g. DR001): ").strip()
        get_safety_score(pid, did)
    elif choice == "3":
        pid = input("  Patient ID (e.g. P001): ").strip()
        did = input("  Drug ID   (e.g. DR001): ").strip()
        get_genomic_compatibility(pid, did)


def menu_procedure():
    print("\n--- PROCEDURE: Generate_Log_Entry ---")
    log_id  = input("  New Log ID     (e.g. L999)  : ").strip()
    pid     = input("  Patient ID     (e.g. P010)  : ").strip()
    did     = input("  Doctor ID      (e.g. D004)  : ").strip()
    hr      = int(input("  Heart Rate     (e.g. 78)    : ").strip())
    temp    = float(input("  Temperature    (e.g. 37.1)  : ").strip())
    bp      = input("  Blood Pressure (e.g. 120/80): ").strip()
    outcome = input("  Outcome (Stable/Improved/Fully recovered/Monitoring required/Under observation): ").strip()
    generate_log_entry(log_id, pid, did, hr, temp, bp, outcome)


def menu_cursor():
    print("\n--- CURSOR: Review_Side_Effects ---")
    pid = input("  Patient ID (e.g. P001): ").strip()
    review_side_effects(pid)


def menu_triggers():
    print("\n--- TRIGGER DEMOS ---")
    print("1. Block future DOB")
    print("2. Block future validation date")
    print("3. GENOMIC_DATA_MISSING exception")
    print("4. Safety Alert on Clinical Log (NEW)")
    choice = input("Choice: ").strip()

    if choice == "1":
        print("\nAttempting to insert patient with DOB = 2099...")
        dob = "2099-01-01"
        if dob > date.today().isoformat():
            print("TRIGGER FIRED: Date of birth cannot be in the future.")

    elif choice == "2":
        print("\nAttempting validation with date = 2099...")
        vdate = "2099-01-01"
        if vdate > date.today().isoformat():
            print("TRIGGER FIRED: Validation date cannot be in the future.")

    elif choice == "3":
        print("\nCreating patient P999 with no genomic profile...")
        try:
            cursor.execute(
                "INSERT OR IGNORE INTO Patient VALUES ('P999','Test','User','1990-01-01','O+')"
            )
            conn.commit()
        except:
            pass
        print("Calling Generate_Log_Entry for P999...")
        generate_log_entry('L990', 'P999', 'D001', 80, 37.0, '120/80', 'Stable')

    elif choice == "4":
        print("\nTrying to log entry for a patient with Blocked validation...")
        print("   Using P001 who has CYP2D6*6 — Blocked for CYP2D6 drugs")
        generate_log_entry('L991', 'P001', 'D005', 75, 37.1, '118/78', 'Stable')


def menu_views():
    print("\n--- VIEWS ---")
    print("1.  Variant Distribution")
    print("2.  Top 10 Symptoms")
    print("3.  High Risk Patients")
    print("4.  Doctor Workload")
    print("5.  Drug Safety Summary")
    print("6.  Blocked Prescriptions")
    print("7.  Patient Clinical Summary")
    print("8.  Drug Side Effect Profile")
    print("9.  Patient Drug Compatibility (NEW)")
    print("10. Safety Validation Detail")
    choice = input("Choice: ").strip()

    queries = {
        "1": ("Variant Distribution", """
            SELECT variant, COUNT(*) AS PatientCount,
                   ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM Genomic_Profile),2) AS Percentage
            FROM Genomic_Profile GROUP BY variant ORDER BY PatientCount DESC"""),

        "2": ("Top 10 Symptoms", """
            SELECT symptom, COUNT(*) AS Frequency
            FROM Log_Symptoms GROUP BY symptom
            ORDER BY Frequency DESC LIMIT 10"""),

        "3": ("High Risk Patients", """
            SELECT p.patient_id,
                   p.first_name||' '||p.last_name AS PatientName,
                   gp.variant, COUNT(*) AS BlockedCount
            FROM Patient p
            JOIN Genomic_Profile gp ON p.patient_id = gp.patient_id
            JOIN Safety_Validation sv ON p.patient_id = sv.patient_id
            WHERE sv.status = 'Blocked'
            GROUP BY p.patient_id ORDER BY BlockedCount DESC LIMIT 10"""),

        "4": ("Doctor Workload", """
            SELECT d.first_name||' '||d.last_name AS DoctorName,
                   d.specialization,
                   COUNT(DISTINCT sv.validation_id) AS TotalValidations,
                   SUM(CASE WHEN sv.status='Blocked' THEN 1 ELSE 0 END) AS Blocked,
                   SUM(CASE WHEN sv.status='Caution' THEN 1 ELSE 0 END) AS Caution,
                   SUM(CASE WHEN sv.status='Safe'    THEN 1 ELSE 0 END) AS Safe
            FROM Doctor d
            LEFT JOIN Safety_Validation sv ON d.doctor_id = sv.doctor_id
            GROUP BY d.doctor_id ORDER BY Blocked DESC"""),

        "5": ("Drug Safety Summary", """
            SELECT m.name AS DrugName,
                   SUM(CASE WHEN sv.status='Blocked' THEN 1 ELSE 0 END) AS Blocked,
                   SUM(CASE WHEN sv.status='Caution' THEN 1 ELSE 0 END) AS Caution,
                   SUM(CASE WHEN sv.status='Safe'    THEN 1 ELSE 0 END) AS Safe
            FROM Medication m
            LEFT JOIN Safety_Validation sv ON m.drug_id = sv.drug_id
            GROUP BY m.drug_id ORDER BY Blocked DESC"""),

        "6": ("Blocked Prescriptions", """
            SELECT sv.validation_id, sv.validation_date,
                   p.first_name||' '||p.last_name AS Patient,
                   gp.variant,
                   d.first_name||' '||d.last_name AS Doctor,
                   m.name AS Drug, gc.reason
            FROM Safety_Validation sv
            JOIN Patient p ON sv.patient_id = p.patient_id
            JOIN Doctor d ON sv.doctor_id = d.doctor_id
            JOIN Medication m ON sv.drug_id = m.drug_id
            JOIN Genomic_Profile gp ON sv.patient_id = gp.patient_id
            LEFT JOIN Genomic_Drug_Compatibility gc
                ON gc.variant = gp.variant AND gc.gene_pathway = m.gene_pathway
            WHERE sv.status = 'Blocked' LIMIT 15"""),

        "7": ("Patient Clinical Summary", """
            SELECT p.first_name||' '||p.last_name AS PatientName,
                   COUNT(cl.log_id) AS TotalVisits,
                   ROUND(AVG(cl.heart_rate),1) AS AvgHeartRate,
                   ROUND(AVG(cl.temperature),2) AS AvgTemp,
                   MAX(cl.log_date) AS LastVisit
            FROM Patient p
            LEFT JOIN Clinical_Log cl ON p.patient_id = cl.patient_id
            GROUP BY p.patient_id
            ORDER BY TotalVisits DESC LIMIT 10"""),

        "8": ("Drug Side Effect Profile", """
            SELECT m.name AS DrugName, m.gene_pathway,
                   mse.side_effect
            FROM Medication m
            JOIN Medication_Side_Effects mse ON m.drug_id = mse.drug_id
            ORDER BY m.name"""),

        "9": ("Patient Drug Compatibility", """
            SELECT p.patient_id,
                   p.first_name||' '||p.last_name AS PatientName,
                   gp.variant, m.name AS DrugName,
                   m.gene_pathway, m.standard_dose,
                   COALESCE(gc.compatibility, 'Safe') AS RecommendedStatus,
                   CASE
                       WHEN gc.compatibility = 'Blocked' THEN 'DO NOT ADMINISTER'
                       WHEN gc.compatibility = 'Caution'
                           THEN 'Reduce dose from '||m.standard_dose
                       ELSE m.standard_dose
                   END AS RecommendedDose,
                   COALESCE(gc.reason, 'Normal metabolizer') AS Reason
            FROM Patient p
            JOIN Genomic_Profile gp ON p.patient_id = gp.patient_id
            JOIN Medication m
            LEFT JOIN Genomic_Drug_Compatibility gc
                ON gc.variant = gp.variant AND gc.gene_pathway = m.gene_pathway
            WHERE p.patient_id = ?
            ORDER BY RecommendedStatus DESC"""),

        "10": ("Safety Validation Detail", """
            SELECT sv.validation_id, sv.validation_date, sv.status,
                   p.first_name||' '||p.last_name AS Patient,
                   d.first_name||' '||d.last_name AS Doctor,
                   m.name AS Drug
            FROM Safety_Validation sv
            JOIN Patient p ON sv.patient_id = p.patient_id
            JOIN Doctor d ON sv.doctor_id = d.doctor_id
            JOIN Medication m ON sv.drug_id = m.drug_id
            LIMIT 15"""),
    }

    if choice in queries:
        title, query = queries[choice]
        if choice == "9":
            pid = input("  Patient ID (e.g. P001): ").strip()
            run_query(query, (pid,), title=title)
        else:
            run_query(query, title=title)
    else:
        print("Invalid choice")


def menu_analytics():
    print("\n--- ANALYTICAL QUERIES ---")
    print("1. Most dangerous drug")
    print("2. Most common genomic variant")
    print("3. Doctors with most blocked cases")
    print("4. Patients with most blocked validations")
    print("5. Most frequent symptom per outcome")
    print("6. Compatibility rules summary (NEW)")
    print("7. Patients safe for all drugs (NEW)")
    choice = input("Choice: ").strip()

    queries = {
        "1": ("Most Dangerous Drug", """
            SELECT m.name AS Drug, COUNT(*) AS BlockedCount
            FROM Safety_Validation sv
            JOIN Medication m ON sv.drug_id = m.drug_id
            WHERE sv.status = 'Blocked'
            GROUP BY m.name ORDER BY BlockedCount DESC"""),

        "2": ("Most Common Genomic Variant", """
            SELECT variant, COUNT(*) AS Count
            FROM Genomic_Profile
            GROUP BY variant ORDER BY Count DESC"""),

        "3": ("Doctors With Most Blocked Cases", """
            SELECT d.first_name||' '||d.last_name AS Doctor,
                   d.specialization, COUNT(*) AS BlockedCases
            FROM Safety_Validation sv
            JOIN Doctor d ON sv.doctor_id = d.doctor_id
            WHERE sv.status = 'Blocked'
            GROUP BY d.doctor_id ORDER BY BlockedCases DESC"""),

        "4": ("Patients With Most Blocked Validations", """
            SELECT p.first_name||' '||p.last_name AS Patient,
                   gp.variant, COUNT(*) AS BlockedCount
            FROM Safety_Validation sv
            JOIN Patient p ON sv.patient_id = p.patient_id
            JOIN Genomic_Profile gp ON p.patient_id = gp.patient_id
            WHERE sv.status = 'Blocked'
            GROUP BY p.patient_id ORDER BY BlockedCount DESC LIMIT 10"""),

        "5": ("Most Frequent Symptom Per Outcome", """
            SELECT cl.outcome, ls.symptom, COUNT(*) AS Frequency
            FROM Clinical_Log cl
            JOIN Log_Symptoms ls ON cl.log_id = ls.log_id
            GROUP BY cl.outcome, ls.symptom
            ORDER BY cl.outcome, Frequency DESC"""),

        "6": ("Compatibility Rules Summary", """
            SELECT variant, gene_pathway, compatibility, reason
            FROM Genomic_Drug_Compatibility
            ORDER BY compatibility DESC, variant"""),

        "7": ("Patients Safe for All Drugs", """
            SELECT p.patient_id,
                   p.first_name||' '||p.last_name AS PatientName,
                   gp.variant
            FROM Patient p
            JOIN Genomic_Profile gp ON p.patient_id = gp.patient_id
            WHERE gp.variant = 'Normal'
            ORDER BY p.patient_id"""),
    }

    if choice in queries:
        title, query = queries[choice]
        run_query(query, title=title)
    else:
        print("Invalid choice")


# MAIN MENU
def main_menu():
    while True:
        print("\n" + "="*52)
        print("        BIO SYNC MEDICINE VAULT")
        print("         Database Demo Interface")
        print("="*52)
        print("  1.  Functions   (Age, Safety Score, Compatibility)")
        print("  2.  Procedure   (Generate_Log_Entry)")
        print("  3.  Cursor      (Review_Side_Effects)")
        print("  4.  Triggers    (Live Demo)")
        print("  5.  Views       (10 reports)")
        print("  6.  Analytics   (7 insights)")
        print("  0.  Exit")
        print("="*52)

        choice = input("  Enter choice: ").strip()

        if   choice == "1": menu_functions()
        elif choice == "2": menu_procedure()
        elif choice == "3": menu_cursor()
        elif choice == "4": menu_triggers()
        elif choice == "5": menu_views()
        elif choice == "6": menu_analytics()
        elif choice == "0":
            print("\nGoodbye! Good luck with your viva!")
            break
        else:
            print("Invalid choice. Try again.")

# Start
main_menu()


Connected to biosync.db

        BIO SYNC MEDICINE VAULT
         Database Demo Interface
  1.  Functions   (Age, Safety Score, Compatibility)
  2.  Procedure   (Generate_Log_Entry)
  3.  Cursor      (Review_Side_Effects)
  4.  Triggers    (Live Demo)
  5.  Views       (10 reports)
  6.  Analytics   (7 insights)
  0.  Exit

--- TRIGGER DEMOS ---
1. Block future DOB
2. Block future validation date
3. GENOMIC_DATA_MISSING exception
4. Safety Alert on Clinical Log (NEW)

Attempting to insert patient with DOB = 2099...
TRIGGER FIRED: Date of birth cannot be in the future.

        BIO SYNC MEDICINE VAULT
         Database Demo Interface
  1.  Functions   (Age, Safety Score, Compatibility)
  2.  Procedure   (Generate_Log_Entry)
  3.  Cursor      (Review_Side_Effects)
  4.  Triggers    (Live Demo)
  5.  Views       (10 reports)
  6.  Analytics   (7 insights)
  0.  Exit


In [ ]:
!pip install flask flask-cors pyngrok

In [ ]:
#Flask REST API (Requires Cell 2 to have been run first (database loaded))
from flask import Flask, jsonify, request
from flask_cors import CORS
from pyngrok import ngrok
import sqlite3
import threading

app = Flask(__name__)
CORS(app)

# DB HELPER
def get_db():
    conn = sqlite3.connect('biosync.db')
    conn.row_factory = sqlite3.Row  # returns dict-like rows
    return conn

def query_db(sql, params=None):
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute(sql, params or ())
    rows = cursor.fetchall()
    conn.close()
    return [dict(row) for row in rows]

def execute_db(sql, params=None):
    conn = get_db()
    cursor = conn.cursor()
    try:
        cursor.execute(sql, params or ())
        conn.commit()
        return cursor.rowcount
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

# ROOT
@app.route('/')
def home():
    return jsonify({
        "project"  : "Bio Sync Medicine Vault",
        "version"  : "1.0",
        "endpoints": [
            "/patients",
            "/patients/<id>",
            "/patients/<id>/age",
            "/patients/<id>/clinical-logs",
            "/patients/<id>/compatibility",
            "/doctors",
            "/doctors/<id>/workload",
            "/medications",
            "/medications/<id>/side-effects",
            "/safety/score/<patient_id>/<drug_id>",
            "/safety/blocked",
            "/safety/high-risk",
            "/safety/validation  [POST]",
            "/clinical-log/generate  [POST]",
            "/cursor/side-effects/<patient_id>",
            "/analytics/variants",
            "/analytics/symptoms",
            "/analytics/compatibility-rules",
            "/analytics/safe-patients"
        ]
    })

# PATIENT ROUTES
@app.route('/patients')
def get_patients():
    rows = query_db("SELECT * FROM Patient ORDER BY patient_id")
    return jsonify(rows)


@app.route('/patients/<patient_id>')
def get_patient(patient_id):
    rows = query_db("""
        SELECT p.*,
               g.profile_id, g.seq_id, g.variant, g.test_date
        FROM Patient p
        LEFT JOIN Genomic_Profile g ON p.patient_id = g.patient_id
        WHERE p.patient_id = ?
    """, (patient_id,))
    if not rows:
        return jsonify({"error": "Patient not found"}), 404
    return jsonify(rows[0])


@app.route('/patients/<patient_id>/age')
def get_patient_age(patient_id):
    from datetime import date
    rows = query_db(
        "SELECT first_name, last_name, dob FROM Patient WHERE patient_id=?",
        (patient_id,)
    )
    if not rows:
        return jsonify({"error": "Patient not found"}), 404
    dob = date.fromisoformat(rows[0]['dob'])
    age = (date.today() - dob).days // 365
    return jsonify({
        "patient_id" : patient_id,
        "name"       : f"{rows[0]['first_name']} {rows[0]['last_name']}",
        "dob"        : rows[0]['dob'],
        "age"        : age
    })


@app.route('/patients/<patient_id>/clinical-logs')
def get_patient_logs(patient_id):
    rows = query_db("""
        SELECT cl.log_id, cl.log_date, cl.heart_rate,
               cl.temperature, cl.blood_pressure, cl.outcome,
               cl.doctor_id,
               GROUP_CONCAT(ls.symptom, ', ') AS symptoms
        FROM Clinical_Log cl
        LEFT JOIN Log_Symptoms ls ON cl.log_id = ls.log_id
        WHERE cl.patient_id = ?
        GROUP BY cl.log_id
        ORDER BY cl.log_date DESC
    """, (patient_id,))
    return jsonify(rows)


@app.route('/patients/<patient_id>/compatibility')
def get_patient_compatibility(patient_id):
    rows = query_db("""
        SELECT p.patient_id,
               p.first_name||' '||p.last_name AS PatientName,
               gp.variant,
               m.drug_id, m.name AS DrugName,
               m.gene_pathway, m.standard_dose,
               COALESCE(gc.compatibility, 'Safe') AS RecommendedStatus,
               CASE
                   WHEN gc.compatibility = 'Blocked' THEN 'DO NOT ADMINISTER'
                   WHEN gc.compatibility = 'Caution'
                       THEN 'Reduce dose from '||m.standard_dose
                   ELSE m.standard_dose
               END AS RecommendedDose,
               COALESCE(gc.reason, 'Normal metabolizer') AS Reason
        FROM Patient p
        JOIN Genomic_Profile gp ON p.patient_id = gp.patient_id
        JOIN Medication m
        LEFT JOIN Genomic_Drug_Compatibility gc
            ON gc.variant = gp.variant
            AND gc.gene_pathway = m.gene_pathway
        WHERE p.patient_id = ?
        ORDER BY RecommendedStatus DESC
    """, (patient_id,))
    return jsonify(rows)


# DOCTOR ROUTES
@app.route('/doctors')
def get_doctors():
    rows = query_db("SELECT * FROM Doctor ORDER BY doctor_id")
    return jsonify(rows)


@app.route('/doctors/<doctor_id>/workload')
def get_doctor_workload(doctor_id):
    rows = query_db("""
        SELECT d.doctor_id,
               d.first_name||' '||d.last_name AS DoctorName,
               d.specialization,
               COUNT(DISTINCT sv.validation_id) AS TotalValidations,
               SUM(CASE WHEN sv.status='Blocked' THEN 1 ELSE 0 END) AS Blocked,
               SUM(CASE WHEN sv.status='Caution' THEN 1 ELSE 0 END) AS Caution,
               SUM(CASE WHEN sv.status='Safe'    THEN 1 ELSE 0 END) AS Safe,
               (SELECT COUNT(*) FROM Clinical_Log cl
                WHERE cl.doctor_id = d.doctor_id) AS ClinicalEntries
        FROM Doctor d
        LEFT JOIN Safety_Validation sv ON d.doctor_id = sv.doctor_id
        WHERE d.doctor_id = ?
        GROUP BY d.doctor_id
    """, (doctor_id,))
    if not rows:
        return jsonify({"error": "Doctor not found"}), 404
    return jsonify(rows[0])


# MEDICATION ROUTES
@app.route('/medications')
def get_medications():
    rows = query_db("SELECT * FROM Medication ORDER BY drug_id")
    return jsonify(rows)


@app.route('/medications/<drug_id>/side-effects')
def get_side_effects(drug_id):
    rows = query_db("""
        SELECT m.drug_id, m.name AS DrugName,
               m.gene_pathway, m.standard_dose,
               mse.side_effect
        FROM Medication m
        JOIN Medication_Side_Effects mse ON m.drug_id = mse.drug_id
        WHERE m.drug_id = ?
    """, (drug_id,))
    return jsonify(rows)


# SAFETY ROUTES
@app.route('/safety/score/<patient_id>/<drug_id>')
def get_safety_score(patient_id, drug_id):
    rows = query_db("""
        SELECT sv.status, m.name AS drug, gp.variant
        FROM Safety_Validation sv
        JOIN Medication m ON sv.drug_id = m.drug_id
        JOIN Genomic_Profile gp ON sv.patient_id = gp.patient_id
        WHERE sv.patient_id = ? AND sv.drug_id = ?
        ORDER BY sv.validation_date DESC LIMIT 1
    """, (patient_id, drug_id))

    score_map = {'Safe': 1, 'Caution': 2, 'Blocked': 3}
    if not rows:
        return jsonify({
            "patient_id"   : patient_id,
            "drug_id"      : drug_id,
            "safety_score" : -1,
            "status"       : "No Record"
        })
    status = rows[0]['status']
    return jsonify({
        "patient_id"   : patient_id,
        "drug_id"      : drug_id,
        "drug_name"    : rows[0]['drug'],
        "variant"      : rows[0]['variant'],
        "safety_score" : score_map.get(status, -1),
        "status"       : status
    })


@app.route('/safety/blocked')
def get_blocked():
    rows = query_db("""
        SELECT sv.validation_id, sv.validation_date,
               p.first_name||' '||p.last_name AS Patient,
               gp.variant,
               d.first_name||' '||d.last_name AS Doctor,
               m.name AS Drug, gc.reason
        FROM Safety_Validation sv
        JOIN Patient p ON sv.patient_id = p.patient_id
        JOIN Doctor d ON sv.doctor_id = d.doctor_id
        JOIN Medication m ON sv.drug_id = m.drug_id
        JOIN Genomic_Profile gp ON sv.patient_id = gp.patient_id
        LEFT JOIN Genomic_Drug_Compatibility gc
            ON gc.variant = gp.variant
            AND gc.gene_pathway = m.gene_pathway
        WHERE sv.status = 'Blocked'
        LIMIT 50
    """)
    return jsonify(rows)


@app.route('/safety/high-risk')
def get_high_risk():
    rows = query_db("""
        SELECT p.patient_id,
               p.first_name||' '||p.last_name AS PatientName,
               gp.variant, COUNT(*) AS BlockedCount
        FROM Safety_Validation sv
        JOIN Patient p ON sv.patient_id = p.patient_id
        JOIN Genomic_Profile gp ON p.patient_id = gp.patient_id
        WHERE sv.status = 'Blocked'
        GROUP BY p.patient_id
        ORDER BY BlockedCount DESC
    """)
    return jsonify(rows)


@app.route('/safety/validation', methods=['POST'])
def add_validation():
    data = request.get_json()
    required = ['validation_id','validation_date','status',
                'patient_id','doctor_id','drug_id']
    for field in required:
        if field not in data:
            return jsonify({"error": f"Missing field: {field}"}), 400
    try:
        execute_db("""
            INSERT INTO Safety_Validation
            VALUES (?,?,?,?,?,?)
        """, (
            data['validation_id'], data['validation_date'],
            data['status'], data['patient_id'],
            data['doctor_id'], data['drug_id']
        ))
        return jsonify({"message": "Validation added successfully"}), 201
    except Exception as e:
        return jsonify({"error": str(e)}), 400


# PROCEDURE ROUTE
@app.route('/clinical-log/generate', methods=['POST'])
def generate_log():
    from datetime import date
    data = request.get_json()
    required = ['log_id','patient_id','doctor_id',
                'heart_rate','temperature','blood_pressure','outcome']
    for field in required:
        if field not in data:
            return jsonify({"error": f"Missing field: {field}"}), 400

    pid = data['patient_id']
    did = data['doctor_id']

    # Check patient
    rows = query_db("SELECT COUNT(*) AS c FROM Patient WHERE patient_id=?", (pid,))
    if rows[0]['c'] == 0:
        return jsonify({"error": "EXCEPTION: Patient not found"}), 404

    # Check doctor
    rows = query_db("SELECT COUNT(*) AS c FROM Doctor WHERE doctor_id=?", (did,))
    if rows[0]['c'] == 0:
        return jsonify({"error": "EXCEPTION: Doctor not found"}), 404

    # GENOMIC_DATA_MISSING
    rows = query_db(
        "SELECT COUNT(*) AS c FROM Genomic_Profile WHERE patient_id=?", (pid,)
    )
    if rows[0]['c'] == 0:
        return jsonify({
            "error": "GENOMIC_DATA_MISSING: No genomic profile for this patient"
        }), 400

    # Duplicate check
    rows = query_db(
        "SELECT COUNT(*) AS c FROM Clinical_Log WHERE log_id=?", (data['log_id'],)
    )
    if rows[0]['c'] > 0:
        return jsonify({"error": "EXCEPTION: Duplicate log_id"}), 400

    # Safety trigger
    rows = query_db("""
        SELECT COUNT(*) AS c FROM Safety_Validation
        WHERE patient_id=? AND status='Blocked'
    """, (pid,))
    if rows[0]['c'] > 0:
        return jsonify({
            "error": "SAFETY ALERT: Blocked validation exists for this patient"
        }), 403

    try:
        execute_db("""
            INSERT INTO Clinical_Log
            (log_id, log_date, heart_rate, temperature,
             blood_pressure, outcome, patient_id, doctor_id)
            VALUES (?, DATE('now'), ?, ?, ?, ?, ?, ?)
        """, (
            data['log_id'], data['heart_rate'], data['temperature'],
            data['blood_pressure'], data['outcome'],
            data['patient_id'], data['doctor_id']
        ))
        return jsonify({
            "message": f"Log {data['log_id']} created for patient {pid}"
        }), 201
    except Exception as e:
        return jsonify({"error": str(e)}), 400
# CURSOR ROUTE
@app.route('/cursor/side-effects/<patient_id>')
def cursor_side_effects(patient_id):
    rows = query_db("""
        SELECT DISTINCT m.drug_id, m.name AS DrugName,
               mse.side_effect AS SideEffect,
               sv.status AS RiskStatus
        FROM Safety_Validation sv
        JOIN Medication m ON sv.drug_id = m.drug_id
        JOIN Medication_Side_Effects mse ON m.drug_id = mse.drug_id
        WHERE sv.patient_id = ?
        ORDER BY sv.status DESC, m.name
    """, (patient_id,))
    return jsonify({
        "patient_id" : patient_id,
        "total"      : len(rows),
        "results"    : rows
    })
# ANALYTICS ROUTES
@app.route('/analytics/variants')
def variant_distribution():
    rows = query_db("""
        SELECT variant, COUNT(*) AS PatientCount,
               ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM Genomic_Profile),2) AS Percentage
        FROM Genomic_Profile
        GROUP BY variant ORDER BY PatientCount DESC
    """)
    return jsonify(rows)


@app.route('/analytics/symptoms')
def symptom_frequency():
    rows = query_db("""
        SELECT symptom, COUNT(*) AS Frequency
        FROM Log_Symptoms
        GROUP BY symptom ORDER BY Frequency DESC
    """)
    return jsonify(rows)


@app.route('/analytics/compatibility-rules')
def compatibility_rules():
    rows = query_db("""
        SELECT variant, gene_pathway, compatibility, reason
        FROM Genomic_Drug_Compatibility
        ORDER BY compatibility DESC, variant
    """)
    return jsonify(rows)


@app.route('/analytics/safe-patients')
def safe_patients():
    rows = query_db("""
        SELECT p.patient_id,
               p.first_name||' '||p.last_name AS PatientName,
               gp.variant
        FROM Patient p
        JOIN Genomic_Profile gp ON p.patient_id = gp.patient_id
        WHERE gp.variant = 'Normal'
        ORDER BY p.patient_id
    """)
    return jsonify(rows)
# START FLASK + NGROK
# Set your ngrok token here
ngrok.set_auth_token("3CKyhHLFaeKBmAxYG8GGGYORzZ2_3pXFb6Pq6r2QsV3xSskte")

def run_flask():
    app.run(port=5000, use_reloader=False)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()
public_url = ngrok.connect(5000)
BASE = public_url.public_url
print("\n" + "="*55)
print(" BIO SYNC MEDICINE VAULT API IS LIVE!")
print("="*55)
print(f"\n  Base URL: {BASE}")
print("\n ENDPOINTS TO TEST IN BROWSER:")
print(f"\n  Patients:")
print(f"    {BASE}/patients")
print(f"    {BASE}/patients/P001")
print(f"    {BASE}/patients/P001/age")
print(f"    {BASE}/patients/P001/clinical-logs")
print(f"    {BASE}/patients/P001/compatibility")
print(f"\n  Doctors:")
print(f"    {BASE}/doctors")
print(f"    {BASE}/doctors/D001/workload")
print(f"\n  Medications:")
print(f"    {BASE}/medications")
print(f"    {BASE}/medications/DR001/side-effects")
print(f"\n  Safety:")
print(f"    {BASE}/safety/score/P001/DR003")
print(f"    {BASE}/safety/blocked")
print(f"    {BASE}/safety/high-risk")
print(f"\n  Procedures & Cursors:")
print(f"    {BASE}/cursor/side-effects/P001")
print(f"\n  Analytics:")
print(f"    {BASE}/analytics/variants")
print(f"    {BASE}/analytics/symptoms")
print(f"    {BASE}/analytics/compatibility-rules")
print(f"    {BASE}/analytics/safe-patients")
print("\n" + "="*55)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit



 BIO SYNC MEDICINE VAULT API IS LIVE!

  Base URL: https://puppet-expansion-gender.ngrok-free.dev

 ENDPOINTS TO TEST IN BROWSER:

  Patients:
    https://puppet-expansion-gender.ngrok-free.dev/patients
    https://puppet-expansion-gender.ngrok-free.dev/patients/P001
    https://puppet-expansion-gender.ngrok-free.dev/patients/P001/age
    https://puppet-expansion-gender.ngrok-free.dev/patients/P001/clinical-logs
    https://puppet-expansion-gender.ngrok-free.dev/patients/P001/compatibility

  Doctors:
    https://puppet-expansion-gender.ngrok-free.dev/doctors
    https://puppet-expansion-gender.ngrok-free.dev/doctors/D001/workload

  Medications:
    https://puppet-expansion-gender.ngrok-free.dev/medications
    https://puppet-expansion-gender.ngrok-free.dev/medications/DR001/side-effects

  Safety:
    https://puppet-expansion-gender.ngrok-free.dev/safety/score/P001/DR003
    https://puppet-expansion-gender.ngrok-free.dev/safety/blocked
    https://puppet-expansion-gender.ngrok-free.